# 03 — Model Experiments

Why XGBoost with these hyperparameters, and how it compares to simpler baselines.

**The honest headline:** the shipped model reaches **AUC-ROC 0.729** against a project target
of 0.80. This notebook establishes that the gap is a *data* ceiling, not a modelling mistake —
several model families land within a few points of each other, which is the signature of a
weak signal rather than an under-tuned estimator.

In [ ]:
import sys, pathlib

root = pathlib.Path.cwd()
while root != root.parent and not (root / 'app' / 'etl').exists():
    root = root.parent
sys.path.insert(0, str(root))
print('ai-service root:', root)

import json
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style='whitegrid')
pd.set_option('display.width', 140)

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    roc_auc_score, roc_curve, precision_recall_curve, average_precision_score,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
)
from xgboost import XGBClassifier

from app.etl.transform import ALL_FEATURES

RANDOM_STATE = 42

## Data

The processed dataset written by the ETL pipeline — already cleaned, engineered, and scaled.

In [ ]:
proc = pd.read_csv(root / 'data' / 'processed' / 'hr_processed.csv')
X = proc[ALL_FEATURES]
y = proc['attrition']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(f'train {X_train.shape}   test {X_test.shape}')
print(f'attrition rate — train {y_train.mean():.2%}, test {y_test.mean():.2%}')

# Imbalance correction, same value the training script uses.
spw = (y_train == 0).sum() / (y_train == 1).sum()
print(f'scale_pos_weight = {spw:.2f}')

## Baselines first

No model is worth shipping until it beats these. The majority-class baseline is the
accuracy trap: it scores well and predicts nothing useful (AUC 0.5).

In [ ]:
def evaluate(name, model, fit=True):
    if fit:
        model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)
    return {
        'model': name,
        'auc_roc': roc_auc_score(y_test, proba),
        'avg_precision': average_precision_score(y_test, proba),
        'accuracy': accuracy_score(y_test, pred),
        'precision': precision_score(y_test, pred, zero_division=0),
        'recall': recall_score(y_test, pred, zero_division=0),
        'f1': f1_score(y_test, pred, zero_division=0),
    }, proba

results, probas = [], {}

for name, mdl in [
    ('Majority class', DummyClassifier(strategy='most_frequent')),
    ('Stratified random', DummyClassifier(strategy='stratified', random_state=RANDOM_STATE)),
]:
    row, p = evaluate(name, mdl)
    results.append(row); probas[name] = p

pd.DataFrame(results).set_index('model').round(4)

## Model families

Each gets the same imbalance handling so the comparison is fair.

In [ ]:
candidates = {
    'Logistic regression': LogisticRegression(max_iter=1000, class_weight='balanced',
                                              random_state=RANDOM_STATE),
    'Decision tree (d=5)': DecisionTreeClassifier(max_depth=5, class_weight='balanced',
                                                  random_state=RANDOM_STATE),
    'Random forest': RandomForestClassifier(n_estimators=300, max_depth=8,
                                            class_weight='balanced', n_jobs=-1,
                                            random_state=RANDOM_STATE),
    'XGBoost (defaults)': XGBClassifier(eval_metric='logloss', scale_pos_weight=spw,
                                        random_state=RANDOM_STATE),
}

for name, mdl in candidates.items():
    row, p = evaluate(name, mdl)
    results.append(row); probas[name] = p

pd.DataFrame(results).set_index('model').round(4).sort_values('auc_roc', ascending=False)

## The shipped configuration

Hyperparameters as persisted in `artifacts/training_metadata.json`. The regularisation
(`gamma`, `reg_alpha`, `reg_lambda`, `min_child_weight`) and subsampling matter more than
depth here — with a weak signal, an unregularised booster memorises noise.

In [ ]:
meta = json.loads((root / 'app' / 'artifacts' / 'training_metadata.json').read_text())
SHIPPED = meta['hyperparameters']
print(json.dumps(SHIPPED, indent=2))

In [ ]:
shipped = XGBClassifier(**{k: v for k, v in SHIPPED.items() if k != 'scale_pos_weight'},
                        scale_pos_weight=spw, random_state=RANDOM_STATE)
row, p = evaluate('XGBoost (shipped)', shipped)
results.append(row); probas['XGBoost (shipped)'] = p

table = pd.DataFrame(results).set_index('model').round(4).sort_values('auc_roc', ascending=False)
table

### Reproduces the persisted metrics?

In [ ]:
print('this run   AUC-ROC:', round(row['auc_roc'], 4))
print('artifact   AUC-ROC:', meta['metrics']['auc_roc'])
print('\nSmall differences are expected — the artifact was trained on its own split.')

## Where the models actually differ

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

for name in ['Logistic regression', 'Random forest', 'XGBoost (shipped)']:
    fpr, tpr, _ = roc_curve(y_test, probas[name])
    axes[0].plot(fpr, tpr, lw=2, label=f'{name} (AUC {roc_auc_score(y_test, probas[name]):.3f})')
axes[0].plot([0, 1], [0, 1], '--', color='#999', label='Chance (0.500)')
axes[0].set_xlabel('False positive rate'); axes[0].set_ylabel('True positive rate')
axes[0].set_title('ROC'); axes[0].legend(loc='lower right')

for name in ['Logistic regression', 'Random forest', 'XGBoost (shipped)']:
    pr, rc, _ = precision_recall_curve(y_test, probas[name])
    axes[1].plot(rc, pr, lw=2, label=f'{name} (AP {average_precision_score(y_test, probas[name]):.3f})')
axes[1].axhline(y_test.mean(), ls='--', color='#999', label=f'Baseline ({y_test.mean():.3f})')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Precision–Recall'); axes[1].legend(loc='upper right')

plt.tight_layout()
plt.show()

### The spread is the finding

Linear, bagged, and boosted models within a few AUC points of each other says the limit is
in the data, not the estimator. A genuinely learnable signal would show boosting pulling
clearly ahead of logistic regression.

In [ ]:
spread = table.loc[['Logistic regression', 'Random forest', 'XGBoost (shipped)'], 'auc_roc']
print(spread)
print(f'\nspread: {spread.max() - spread.min():.4f} AUC')

## Cross-validation

A single hold-out split can flatter or punish by luck. 5-fold stratified CV gives the
stability estimate — a small standard deviation means the score is real, not split noise.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scores = cross_val_score(shipped, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)

print('fold AUCs:', np.round(scores, 4))
print(f'mean {scores.mean():.4f} +/- {scores.std():.4f}')
print(f"artifact: {meta['metrics']['cv_auc_roc_mean']} +/- {meta['metrics']['cv_auc_roc_std']}")

## Threshold choice is a business decision

0.5 is not special. Missing a leaver (false negative) costs 1.5–2× salary; a false positive
costs a manager a 30-minute conversation. The asymmetry argues for a lower threshold and
higher recall — this table is what an HR lead should actually be shown.

In [ ]:
proba = probas['XGBoost (shipped)']
rows = []
for t in np.arange(0.20, 0.75, 0.05):
    pred = (proba >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    rows.append({'threshold': round(t, 2),
                 'flagged': int(pred.sum()),
                 'caught (TP)': int(tp), 'missed (FN)': int(fn),
                 'false alarms (FP)': int(fp),
                 'precision': precision_score(y_test, pred, zero_division=0),
                 'recall': recall_score(y_test, pred, zero_division=0),
                 'f1': f1_score(y_test, pred, zero_division=0)})

pd.DataFrame(rows).set_index('threshold').round(3)

In [ ]:
# Expected cost, given the asymmetry. Illustrative numbers — swap in your own.
COST_MISS = 90_000   # replacing someone who left undetected
COST_FALSE_ALARM = 500   # a retention conversation that was not needed

ts = np.arange(0.05, 0.96, 0.01)
costs = []
for t in ts:
    pred = (proba >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    costs.append(fn * COST_MISS + fp * COST_FALSE_ALARM)

best = ts[int(np.argmin(costs))]
plt.figure(figsize=(9, 5))
plt.plot(ts, costs, lw=2, color='#1677ff')
plt.axvline(best, ls='--', color='#52c41a', label=f'cost-optimal ~{best:.2f}')
plt.axvline(0.50, ls='--', color='#ff4d4f', label='default 0.50')
plt.xlabel('Decision threshold'); plt.ylabel('Expected cost on the test set ($)')
plt.title('Threshold vs expected cost')
plt.legend(); plt.tight_layout(); plt.show()

print(f'Cost-optimal threshold ~{best:.2f} — well below 0.50, because a miss costs',
      f'{COST_MISS / COST_FALSE_ALARM:.0f}x a false alarm.')
print('\nNote: the app buckets at 0.3 / 0.6 for LOW / MEDIUM / HIGH, which already leans',
      'toward recall rather than using a bare 0.5 cut.')

## Confusion matrix at the shipped bucketing

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
for ax, t in zip(axes, [0.30, 0.50]):
    ConfusionMatrixDisplay(
        confusion_matrix(y_test, (proba >= t).astype(int)),
        display_labels=['Stayed', 'Left'],
    ).plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f'threshold = {t:.2f}')
plt.tight_layout(); plt.show()

print(classification_report(y_test, (proba >= 0.30).astype(int),
                            target_names=['Stayed', 'Left']))

## Feature importance

Gain — how much each feature improved the splits that used it. Compare against the Cohen's d
ranking from notebook 01: the engineered ratios earn their place by contributing here even
where their univariate separation was modest.

In [ ]:
imp = (pd.Series(shipped.feature_importances_, index=ALL_FEATURES)
         .sort_values(ascending=False))

plt.figure(figsize=(9, 6))
colours = ['#722ed1' if f not in ALL_FEATURES[:8] else '#1677ff' for f in imp.index[::-1]]
plt.barh(imp.index[::-1], imp.values[::-1], color=colours)
plt.xlabel('Gain'); plt.title('Feature importance — base (blue) vs engineered (purple)')
plt.tight_layout(); plt.show()

print(imp.round(4).to_string())
print('\nArtifact ranking for comparison:')
print(pd.Series(meta['feature_importance']).sort_values(ascending=False).round(4).to_string())

## Does more data help? (learning curve)

The decisive test. If validation AUC has plateaued, more rows of the *same* synthetic data
will not close the gap to 0.80 — the generator's signal is the ceiling.

In [ ]:
from sklearn.model_selection import learning_curve

sizes, train_sc, val_sc = learning_curve(
    shipped, X, y, cv=cv, scoring='roc_auc', n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 8),
)

plt.figure(figsize=(9, 5))
plt.plot(sizes, train_sc.mean(1), 'o-', label='train', color='#1677ff')
plt.fill_between(sizes, train_sc.mean(1) - train_sc.std(1), train_sc.mean(1) + train_sc.std(1),
                 alpha=0.15, color='#1677ff')
plt.plot(sizes, val_sc.mean(1), 'o-', label='validation', color='#52c41a')
plt.fill_between(sizes, val_sc.mean(1) - val_sc.std(1), val_sc.mean(1) + val_sc.std(1),
                 alpha=0.15, color='#52c41a')
plt.axhline(0.80, ls='--', color='#ff4d4f', label='target 0.80')
plt.xlabel('Training rows'); plt.ylabel('AUC-ROC')
plt.title('Learning curve')
plt.legend(); plt.tight_layout(); plt.show()

print('validation AUC by training size:')
for n, s in zip(sizes, val_sc.mean(1)):
    print(f'  {int(n):5d} rows -> {s:.4f}')
print(f'\ngain from the last doubling: {val_sc.mean(1)[-1] - val_sc.mean(1)[-3]:+.4f} AUC')

---
## Conclusions

1. **The model beats every baseline** — AUC 0.729 against 0.500 for chance, and it beats the
   accuracy trap that a majority-class predictor sets.
2. **It falls short of the 0.80 target.** Stated plainly rather than buried.
3. **The ceiling is the data.** Logistic regression, random forest, and tuned XGBoost land
   within a few AUC points of one another, and the validation curve flattens well before
   5,000 rows. That is a weak-signal signature, not an under-tuned model.
4. **Cross-validation is tight** (std ≈ 0.005), so 0.729 is a real estimate, not split luck.
5. **Threshold is a business lever.** Because a miss costs orders of magnitude more than a
   false alarm, the cost-optimal cut sits far below 0.50 — which is why the app's LOW/MEDIUM/HIGH
   buckets break at 0.3 and 0.6.

### To actually reach 0.80

In order of expected payoff:

1. **Use real data** — the IBM HR Attrition dataset, or the organisation's own history. This is
   the only change that lifts the ceiling rather than working under it.
2. **Strengthen the generator** — encode genuine interaction effects (e.g. low engagement is
   far more dangerous when pay is also below band) instead of largely independent noise.
3. **Add features with real predictive power** — manager changes, internal transfers, comp
   ratio against market band, survey free-text sentiment, commute distance.
4. **Only then tune** — hyperparameter search on a weak signal buys fractions of a point and
   invites overfitting to the validation split.